In [ ]:
from pathlib import Path
import json
import pickle
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

PROJECT_DIR = Path('/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC')
SPATIALPYALAB_DIR = Path('/Volumes/B/Projects/2025Project/SpatialAnalysis/Code')
if str(SPATIALPYALAB_DIR) not in sys.path:
    sys.path.append(str(SPATIALPYALAB_DIR))
import SpatialPyALab as lab

sc.set_figure_params(dpi=160, color_map='viridis_r')
sc.settings.verbosity = 1
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'


In [ ]:
LAYER_WIDTH = 20
MAX_OUTWARD_DISTANCE = 500
MIN_LAYER_AREA = 20

CT_COLORS = {
    'Bcell': '#ba55d3',
    'Endothelial': '#1E90FF',
    'Epithelial': '#9fa6ab',
    'Hepatocyte': '#639d63',
    'Fibroblast': '#BD8F8F',
    'Myeloid': '#FFD500',
    'Tcell': '#FF0000',
    'conflict': '#e9e9e9',
}
PROPORTION_PLOTS = {
    'Endothelial': {'point_color': '#1E90FF', 'spot_size': 0.8, 'vmin': 0, 'vmax': 0.25},
    'Fibroblast': {'point_color': '#BD8F8F', 'spot_size': 1, 'vmin': 0.3, 'vmax': 1},
    'Myeloid': {'point_color': '#FFD500', 'spot_size': 1, 'vmin': 0, 'vmax': 0.6},
    'Tcell': {'point_color': '#FF0000', 'spot_size': 1, 'vmin': 0, 'vmax': 0.4},
    'Bcell': {'point_color': '#ba55d3', 'spot_size': 1, 'vmin': 0, 'vmax': 0.3},
}
SAMPLE_CONFIGS = {
    'Pt5-4': {
        'name': 'Pt5-4',
        'sample_id': 'SR250530M3',
        'root_dir': PROJECT_DIR / 'Data/ST/Samples/SR250530M3',
        'region_key': 3,
        'zoom_point_size': 1,
        'zoom_pdf': PROJECT_DIR / 'Figure/ST/Boundary/Figure5D/Pt5-4_layer_plot_region.pdf',
        'myeloid_point_color': '#FFE100',
    },
    'Pt6-5': {
        'name': 'Pt6-5',
        'sample_id': 'SR250521M1',
        'root_dir': PROJECT_DIR / 'Data/ST/Samples/SR250521M1',
        'region_key': 1,
        'zoom_point_size': 2.5,
        'zoom_pdf': PROJECT_DIR / 'Figure/ST/Boundary/Figure5D/Pt6-5_layer_plot_region.pdf',
        'myeloid_point_color': '#FFD500',
    },
}
SAMPLES_TO_RUN = list(SAMPLE_CONFIGS)


In [ ]:
def normalize_cell_ids(values):
    """Normalize identifiers without converting missing values into text labels."""
    values = pd.Series(values, copy=True)
    if values.isna().any():
        raise ValueError('cell_id contains missing values.')
    normalized = values.astype(str).str.strip()
    if normalized.eq('').any():
        raise ValueError('cell_id contains empty values.')
    return normalized


def load_inputs(root_dir):
    """Read the unified cell input, clone polygons, and region mapping."""
    root_dir = Path(root_dir)
    segment_dir = root_dir / 'cell_segment'
    paths = {
        'cells': segment_dir / 'Cell_Segment_addCelltype_adata.h5ad',
        'polygons': segment_dir / 'Region_Clone_Polys_by_GLMPCA.pkl',
        'clone_mapping': segment_dir / 'region_to_clone.json',
    }
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError('Missing input files: ' + ', '.join(missing))
    cell_adata = sc.read_h5ad(paths['cells'])
    required_columns = {'cell_id', 'CellType'}
    missing_columns = required_columns.difference(cell_adata.obs.columns)
    if missing_columns:
        raise ValueError(f'Missing obs columns: {sorted(missing_columns)}')
    if 'spatial_um' not in cell_adata.obsm:
        raise ValueError("Missing obsm['spatial_um'] coordinates.")
    coordinates = np.asarray(cell_adata.obsm['spatial_um'])
    if coordinates.shape != (cell_adata.n_obs, 2):
        raise ValueError("obsm['spatial_um'] must have shape (n_cells, 2).")
    cell_adata = cell_adata[cell_adata.obs['CellType'].astype(str) != 'conflict'].copy()
    cell_adata.obs['cell_id'] = normalize_cell_ids(cell_adata.obs['cell_id']).to_numpy()
    if cell_adata.obs['cell_id'].duplicated().any():
        raise ValueError('Input cell_id values must be unique after normalization.')
    with paths['polygons'].open('rb') as handle:
        clone_polys = pickle.load(handle)
    with paths['clone_mapping'].open('r', encoding='utf-8') as handle:
        region_to_clone = json.load(handle)
    return cell_adata, clone_polys, region_to_clone, paths


def assign_layers(cell_adata, clone_polys, region_to_clone):
    """Assign layers with the original geometry settings and first-hit policy."""
    layers_dict = lab.by.create_layers(
        polys=clone_polys,
        operation_mode='per_poly',
        expansion_mode='both',
        buffer_step=LAYER_WIDTH,
        num_steps=MAX_OUTWARD_DISTANCE // LAYER_WIDTH,
        min_area=MIN_LAYER_AREA,
        consider_others_expansion=True,
        safe_margin=0.0,
        include_center=True,
        sliding_mode=False,
        window_size=None,
        slide_step=None,
        max_dist=None,
    )
    _, assignments = lab.by.assign_points_nonoverlap_fast_from_obsm(
        adata=cell_adata,
        layers_dict=layers_dict,
        obsm_key='spatial_um',
        cell_id_col='cell_id',
        swap_xy=False,
    )
    assignments = assignments.reindex(columns=['cell_id', 'region_key', 'layer']).copy()
    assignments['cell_id'] = normalize_cell_ids(assignments['cell_id']).to_numpy()
    assignments = assignments.drop_duplicates(subset='cell_id', keep='first')
    assignments['clone'] = assignments['region_key'].map(region_to_clone)
    original_index = cell_adata.obs.index.copy()
    stale_columns = ['region_key', 'layer', 'clone', 'distance']
    obs = cell_adata.obs.drop(columns=stale_columns, errors='ignore')
    obs = obs.join(
        assignments.set_index('cell_id'),
        on='cell_id',
        how='left',
        sort=False,
        validate='one_to_one',
    )
    if not obs.index.equals(original_index):
        raise RuntimeError('Layer assignment changed observation names or row order.')
    obs['distance'] = pd.to_numeric(obs['layer'], errors='coerce') * LAYER_WIDTH
    cell_adata.obs = obs
    return cell_adata, layers_dict


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch
from shapely.ops import unary_union

def _clean_geom(geom):
    """Repair invalid geometries with a zero-width buffer when possible."""
    if geom is None:
        return geom
    if geom.is_empty:
        return geom
    try:
        if not geom.is_valid:
            geom = geom.buffer(0)
    except Exception:
        pass
    return geom

def _polygon_to_pathpatch(poly, facecolor, edgecolor='none', linewidth=0, alpha=1.0, zorder=1):
    """Create a PathPatch from a polygon, including its interior holes."""
    vertices = []
    codes = []
    exterior_coords = list(poly.exterior.coords)
    vertices.extend(exterior_coords)
    codes.extend([MplPath.MOVETO] + [MplPath.LINETO] * (len(exterior_coords) - 2) + [MplPath.CLOSEPOLY])
    for interior in poly.interiors:
        interior_coords = list(interior.coords)
        vertices.extend(interior_coords)
        codes.extend([MplPath.MOVETO] + [MplPath.LINETO] * (len(interior_coords) - 2) + [MplPath.CLOSEPOLY])
    path = MplPath(vertices, codes)
    patch = PathPatch(path, facecolor=facecolor, edgecolor=edgecolor, linewidth=linewidth, alpha=alpha, zorder=zorder)
    return patch

def _add_geometry_to_ax(ax, geom, facecolor, edgecolor='none', linewidth=0, alpha=1.0, zorder=1):
    """Add a Polygon or MultiPolygon to the axes."""
    geom = _clean_geom(geom)
    if geom is None or geom.is_empty:
        return
    if geom.geom_type == 'Polygon':
        patch = _polygon_to_pathpatch(geom, facecolor=facecolor, edgecolor=edgecolor, linewidth=linewidth, alpha=alpha, zorder=zorder)
        ax.add_patch(patch)
    elif geom.geom_type == 'MultiPolygon':
        for poly in geom.geoms:
            if poly is None or poly.is_empty:
                continue
            patch = _polygon_to_pathpatch(poly, facecolor=facecolor, edgecolor=edgecolor, linewidth=linewidth, alpha=alpha, zorder=zorder)
            ax.add_patch(patch)

def _draw_original_boundaries(ax, polygons_data, boundary_linewidth=0.35, highlight_layer=None, highlight_linewidth=1.5, line_color='black', zorder=6):
    """Draw original polygon exteriors, optionally highlighting selected layers."""
    for (geom, layer_num) in polygons_data:
        geom = _clean_geom(geom)
        if geom is None or geom.is_empty:
            continue
        layer_num = float(layer_num)
        if highlight_layer is not None:
            if isinstance(highlight_layer, (list, tuple, set)):
                lw = highlight_linewidth if layer_num in highlight_layer else boundary_linewidth
            else:
                lw = highlight_linewidth if layer_num == float(highlight_layer) else boundary_linewidth
        else:
            lw = boundary_linewidth
        if geom.geom_type == 'Polygon':
            (x, y) = geom.exterior.xy
            ax.plot(x, y, color=line_color, linewidth=lw, zorder=zorder)
        elif geom.geom_type == 'MultiPolygon':
            for poly in geom.geoms:
                if poly is None or poly.is_empty:
                    continue
                (x, y) = poly.exterior.xy
                ax.plot(x, y, color=line_color, linewidth=lw, zorder=zorder)

def _build_layer_polygons(polygons_data):
    """Merge geometries that have the same layer number."""
    layer_to_polys = {}
    for (polygon, layer_num) in polygons_data:
        layer_num = float(layer_num)
        polygon = _clean_geom(polygon)
        if polygon is None or polygon.is_empty:
            continue
        if layer_num not in layer_to_polys:
            layer_to_polys[layer_num] = []
        layer_to_polys[layer_num].append(polygon)
    layer_poly = {}
    for (layer_num, polys) in layer_to_polys.items():
        merged = unary_union(polys)
        merged = _clean_geom(merged)
        if merged is not None and (not merged.is_empty):
            layer_poly[layer_num] = merged
    return layer_poly

def _make_rings_for_one_side(layer_poly_side):
    """Convert one side of the layer polygons to rings using the original area-trend rule."""
    if len(layer_poly_side) == 0:
        return []
    layers = sorted(layer_poly_side.keys(), key=lambda x: abs(x))
    areas = np.array([layer_poly_side[x].area for x in layers], dtype=float)
    ring_data = []
    if len(layers) == 1:
        layer = layers[0]
        ring_data.append((layer_poly_side[layer], layer))
        return ring_data
    area_trend = np.nanmedian(np.diff(areas))
    if area_trend >= 0:
        previous_union = None
        for layer in layers:
            poly = layer_poly_side[layer]
            if previous_union is None:
                ring = poly
                previous_union = poly
            else:
                ring = poly.difference(previous_union)
                previous_union = unary_union([previous_union, poly])
            ring = _clean_geom(ring)
            if ring is not None and (not ring.is_empty):
                ring_data.append((ring, layer))
    else:
        for (i, layer) in enumerate(layers):
            poly = layer_poly_side[layer]
            if i < len(layers) - 1:
                next_layer = layers[i + 1]
                next_poly = layer_poly_side[next_layer]
                ring = poly.difference(next_poly)
            else:
                ring = poly
            ring = _clean_geom(ring)
            if ring is not None and (not ring.is_empty):
                ring_data.append((ring, layer))
    return ring_data

def build_layer_rings(polygons_data):
    """Build rings separately for negative, zero, and positive layers."""
    layer_poly = _build_layer_polygons(polygons_data)
    neg_layer_poly = {layer: poly for (layer, poly) in layer_poly.items() if layer < 0}
    zero_layer_poly = {layer: poly for (layer, poly) in layer_poly.items() if layer == 0}
    pos_layer_poly = {layer: poly for (layer, poly) in layer_poly.items() if layer > 0}
    ring_data = []
    ring_data.extend(_make_rings_for_one_side(neg_layer_poly))
    for (layer, poly) in zero_layer_poly.items():
        ring_data.append((poly, layer))
    ring_data.extend(_make_rings_for_one_side(pos_layer_poly))
    ring_data = [(geom, float(layer)) for (geom, layer) in ring_data if geom is not None and (not geom.is_empty)]
    ring_data.sort(key=lambda x: x[1])
    return ring_data

def _weighted_gaussian_smooth_by_x(x, values, weights=None, sigma=1.0):
    """Smooth values along x with Gaussian kernels and optional observation weights."""
    x = np.asarray(x, dtype=float)
    values = np.asarray(values, dtype=float)
    if weights is None:
        weights = np.ones_like(values, dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)
    out = np.full(len(values), np.nan, dtype=float)
    valid = np.isfinite(x) & np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if valid.sum() == 0:
        return out
    for i in range(len(values)):
        if not np.isfinite(x[i]):
            continue
        dist = x[valid] - x[i]
        kernel = np.exp(-0.5 * (dist / sigma) ** 2)
        w = kernel * weights[valid]
        if w.sum() > 0:
            out[i] = np.sum(values[valid] * w) / np.sum(w)
    return out

def smooth_layer_proportions(prop_df, layer_col='layer', value_col='proportion', weight_col='n_total', sigma=1.0, separate_sides=True, keep_zero=True):
    """Preserve the original smoothing: positive layers are smoothed; with keep_zero=True, all nonpositive layers retain their raw values."""
    df = prop_df.copy()
    df = df.sort_values(layer_col).reset_index(drop=True)
    df['proportion_smooth'] = df[value_col].astype(float)
    if not separate_sides:
        x = df[layer_col].values
        y = df[value_col].values
        w = df[weight_col].values if weight_col in df.columns else None
        df['proportion_smooth'] = _weighted_gaussian_smooth_by_x(x=x, values=y, weights=w, sigma=sigma)
        return df
    pos_mask = df[layer_col] > 0
    if pos_mask.any():
        sub = df.loc[pos_mask].sort_values(layer_col)
        df.loc[sub.index, 'proportion_smooth'] = _weighted_gaussian_smooth_by_x(x=sub[layer_col].values, values=sub[value_col].values, weights=sub[weight_col].values if weight_col in sub.columns else None, sigma=sigma)
    zero_mask = df[layer_col] <= 0
    if zero_mask.any():
        if keep_zero:
            df.loc[zero_mask, 'proportion_smooth'] = df.loc[zero_mask, value_col]
        else:
            sub = df.loc[zero_mask]
            df.loc[sub.index, 'proportion_smooth'] = _weighted_gaussian_smooth_by_x(x=sub[layer_col].values, values=sub[value_col].values, weights=sub[weight_col].values if weight_col in sub.columns else None, sigma=sigma)
    return df

def plot_celltype_proportion_in_layers(region_adata, target_celltype, layers_dict, celltype_col='CellType', layer_key='layer', coords_key='spatial_um', cmap='Reds', spot_size=2.5, target_point_color='blue', boundary_linewidth=0.35, highlight_layer=None, highlight_linewidth=1.5, show_colorbar=True, show_target_points=True, target_point_alpha=0.7, figsize=(7, 6), color_range='data', vmin=None, vmax=None, sigma=1.2, polygon_alpha=1.0, use_ring=True, title=None, show_legend=True, draw_original_boundary=True, original_boundary_color='black'):
    """Compute cell-type proportions by layer and plot the original ring geometry and boundaries."""
    obs = region_adata.obs.copy()
    obs[layer_key] = pd.to_numeric(obs[layer_key], errors='coerce')
    layers_in_data = sorted(obs[layer_key].dropna().unique())
    layers_in_data = [float(x) for x in layers_in_data]
    if len(layers_in_data) == 0:
        raise ValueError(f"No valid layer values were found in obs['{layer_key}'].")
    if celltype_col not in obs.columns:
        raise ValueError(f'Cell-type column is missing from obs: {celltype_col}')
    if coords_key not in region_adata.obsm:
        raise ValueError(f'Coordinate key is missing from obsm: {coords_key}')
    rows = []
    for layer in layers_in_data:
        layer_mask = obs[layer_key] == layer
        layer_cells = obs.loc[layer_mask]
        n_total = layer_cells.shape[0]
        n_target = layer_cells[celltype_col].astype(str).eq(target_celltype).sum()
        proportion = n_target / n_total if n_total > 0 else np.nan
        rows.append({'layer': layer, 'n_total': n_total, 'n_target': int(n_target), 'proportion': proportion})
    prop_df = pd.DataFrame(rows)
    prop_df = smooth_layer_proportions(prop_df, layer_col='layer', value_col='proportion', weight_col='n_total', sigma=sigma, separate_sides=True, keep_zero=True)
    proportion_dict = dict(zip(prop_df['layer'], prop_df['proportion_smooth']))
    valid_props = prop_df['proportion_smooth'].replace([np.inf, -np.inf], np.nan).dropna()
    if len(valid_props) == 0:
        raise ValueError('All layer proportions are missing; a color range cannot be set.')
    if color_range == 'fixed':
        vmin_use = 0.0 if vmin is None else float(vmin)
        vmax_use = 1.0 if vmax is None else float(vmax)
    elif color_range == 'data':
        vmin_use = float(valid_props.min()) if vmin is None else float(vmin)
        vmax_use = float(valid_props.max()) if vmax is None else float(vmax)
        if np.isclose(vmin_use, vmax_use):
            vmin_use = max(0.0, vmin_use - 0.05)
            vmax_use = min(1.0, vmax_use + 0.05)
    elif color_range == 'manual':
        if vmin is None or vmax is None:
            raise ValueError("vmin and vmax are required when color_range='manual'.")
        vmin_use = float(vmin)
        vmax_use = float(vmax)
    else:
        raise ValueError("color_range must be 'data', 'fixed', or 'manual'.")
    colormap = plt.cm.get_cmap(cmap)
    norm = Normalize(vmin=vmin_use, vmax=vmax_use)
    (fig, ax) = plt.subplots(1, 1, figsize=figsize)
    ax.set_facecolor('white')
    dict_key = list(layers_dict.keys())[0]
    polygons_data = layers_dict[dict_key]
    if use_ring:
        plot_geom_data = build_layer_rings(polygons_data)
    else:
        plot_geom_data = [(_clean_geom(polygon), float(layer_num)) for (polygon, layer_num) in polygons_data]
    layers_to_plot = []
    for (polygon, layer_num) in plot_geom_data:
        layer_num = float(layer_num)
        if layer_num not in proportion_dict:
            continue
        prop_value = proportion_dict[layer_num]
        if pd.isna(prop_value):
            continue
        layers_to_plot.append((polygon, layer_num, prop_value))
    if len(layers_to_plot) == 0:
        raise ValueError('No polygon layers match the layers in obs.')
    layers_to_plot.sort(key=lambda x: x[1])
    for (polygon, layer_num, prop_value) in layers_to_plot:
        color = colormap(norm(prop_value))
        _add_geometry_to_ax(ax=ax, geom=polygon, facecolor=color, edgecolor='none', linewidth=0, alpha=polygon_alpha, zorder=1)
    if draw_original_boundary:
        _draw_original_boundaries(ax=ax, polygons_data=polygons_data, boundary_linewidth=boundary_linewidth, highlight_layer=highlight_layer, highlight_linewidth=highlight_linewidth, line_color=original_boundary_color, zorder=6)
    if show_target_points:
        ct_mask = obs[celltype_col].astype(str).eq(target_celltype).values
        coords = np.asarray(region_adata.obsm[coords_key])
        if ct_mask.sum() > 0:
            ax.scatter(coords[ct_mask, 0], coords[ct_mask, 1], s=spot_size, c=target_point_color, alpha=target_point_alpha, linewidths=0, label=target_celltype, zorder=10)
    if show_colorbar:
        sm = plt.cm.ScalarMappable(cmap=colormap, norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
        if color_range == 'fixed' and vmin_use == 0 and (vmax_use == 1):
            cbar.set_ticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.relim()
    ax.autoscale_view()
    ax.set_xticks([])
    ax.set_yticks([])
    ax.invert_yaxis()
    ax.set_aspect('equal')
    if title is None:
        title = f'{target_celltype} proportion by layer'
    ax.set_title(title, fontsize=12)
    ax.set_aspect('equal')
    ax.axis('off')
    if show_legend and show_target_points and obs[celltype_col].astype(str).eq(target_celltype).any():
        ax.legend(loc='upper right', frameon=True, fontsize=8, markerscale=2)
    plt.tight_layout()
    return (fig, ax, prop_df)


In [ ]:
def show_spatial_previews(layer_adata):
    """Show the whole-sample cell-type preview without saving an additional file."""
    sc.pl.spatial(
        layer_adata,
        color='CellType',
        basis='spatial_um',
        palette=CT_COLORS,
        spot_size=12,
    )
    plt.close()


def save_region_views(layer_adata, layers_dict, config):
    """Save the zoomed region view with per-sample plotting parameters."""
    region_key = config['region_key']
    polygons_data = layers_dict[region_key]
    coords = np.asarray(layer_adata.obsm['spatial_um'])
    celltype_values = layer_adata.obs['CellType'].astype(str).to_numpy()
    polygon_bounds = np.asarray([
        polygon.bounds for polygon, _ in polygons_data
        if polygon is not None and not polygon.is_empty
    ])
    if len(polygon_bounds) == 0:
        raise ValueError(f'No valid polygons found for region_{region_key}.')
    xmin, ymin = polygon_bounds[:, :2].min(axis=0)
    xmax, ymax = polygon_bounds[:, 2:].max(axis=0)
    x_padding = (xmax - xmin) * 0.05
    y_padding = (ymax - ymin) * 0.05
    view_xmin, view_xmax = xmin - x_padding, xmax + x_padding
    view_ymin, view_ymax = ymin - y_padding, ymax + y_padding
    view_mask = (
        np.isfinite(coords[:, 0])
        & np.isfinite(coords[:, 1])
        & (coords[:, 0] >= view_xmin)
        & (coords[:, 0] <= view_xmax)
        & (coords[:, 1] >= view_ymin)
        & (coords[:, 1] <= view_ymax)
    )
    view_celltypes = sorted(
        layer_adata.obs.loc[view_mask, 'CellType'].dropna().astype(str).unique()
    )
    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    for celltype in view_celltypes:
        mask = view_mask & (celltype_values == celltype)
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            s=config['zoom_point_size'], marker='o',
            c=[CT_COLORS.get(celltype, '#BDBDBD')],
            alpha=1, linewidths=0, label=celltype, zorder=1,
        )
    for polygon, layer_num in polygons_data:
        if polygon is None or polygon.is_empty:
            continue
        lab.by.plot_layers_boundaries(
            layer_adata, {region_key: [(polygon, layer_num)]}, ax=ax,
            edgecolor='black', linewidth=1.5 if float(layer_num) == 0 else 0.1,
        )
    ax.set_xlim(view_xmin, view_xmax)
    ax.set_ylim(view_ymax, view_ymin)
    ax.set_aspect('equal')
    ax.axis('off')
    plt.tight_layout()
    zoom_pdf = Path(config['zoom_pdf'])
    fig.savefig(zoom_pdf, bbox_inches='tight', dpi=300)
    plt.show()
    plt.close(fig)
    return [zoom_pdf]


In [ ]:
def save_proportion_maps(layer_adata, layers_dict, config, figure_dir):
    """Save the five original maps and return their small proportion tables."""
    region_key = config['region_key']
    region_mask = layer_adata.obs['region_key'].astype(str) == f'region_{region_key}'
    region_adata = layer_adata[region_mask].copy()
    exclude = (
        (region_adata.obs['CellType'] == 'Epithelial')
        & (region_adata.obs['distance'] > 0)
    )
    region_adata = region_adata[~exclude].copy()
    if region_adata.n_obs == 0:
        raise ValueError(f'No cells remain for region_{region_key}.')
    target_layers_dict = {region_key: layers_dict[region_key]}
    output_paths = []
    proportion_tables = {}
    for celltype, settings in PROPORTION_PLOTS.items():
        point_color = (
            config['myeloid_point_color'] if celltype == 'Myeloid'
            else settings['point_color']
        )
        fig, ax, prop_df = plot_celltype_proportion_in_layers(
            region_adata=region_adata,
            target_celltype=celltype,
            layers_dict=target_layers_dict,
            celltype_col='CellType',
            layer_key='layer',
            spot_size=settings['spot_size'],
            polygon_alpha=0.7,
            target_point_color=point_color,
            coords_key='spatial_um',
            cmap='OrRd',
            sigma=2,
            boundary_linewidth=0,
            highlight_layer=0,
            highlight_linewidth=0.5,
            show_colorbar=True,
            show_target_points=True,
            figsize=(3.5, 3.5),
            color_range='manual',
            use_ring=True,
            vmin=settings['vmin'],
            vmax=settings['vmax'],
            title='',
            show_legend=True,
            draw_original_boundary=True,
        )
        output_path = Path(figure_dir) / f'{celltype}_layer_proportion.pdf'
        fig.savefig(output_path, bbox_inches='tight', dpi=300, transparent=True)
        plt.show()
        plt.close(fig)
        output_paths.append(output_path)
        proportion_tables[celltype] = prop_df
    return output_paths, proportion_tables


def main(config):
    """Run one sample and return paths and tables without retaining AnnData."""
    root_dir = Path(config['root_dir'])
    cell_adata, clone_polys, region_to_clone, input_paths = load_inputs(root_dir)
    layer_adata, layers_dict = assign_layers(cell_adata, clone_polys, region_to_clone)
    if config['region_key'] not in layers_dict:
        raise KeyError(f"Configured polygon key {config['region_key']} is unavailable.")
    figure_dir = root_dir / 'figure'
    figure_dir.mkdir(parents=True, exist_ok=True)
    Path(config['zoom_pdf']).parent.mkdir(parents=True, exist_ok=True)
    print(f"{config['name']}: {layer_adata.n_obs:,} cells after conflict filtering")
    show_spatial_previews(layer_adata)
    output_paths = save_region_views(layer_adata, layers_dict, config)
    proportion_paths, proportion_tables = save_proportion_maps(
        layer_adata, layers_dict, config, figure_dir
    )
    return {
        'sample_id': config['sample_id'],
        'input_paths': {name: str(path) for name, path in input_paths.items()},
        'output_paths': [str(path) for path in output_paths + proportion_paths],
        'proportions': proportion_tables,
    }


In [ ]:
results = {}
for sample_name in SAMPLES_TO_RUN:
    results[sample_name] = main(SAMPLE_CONFIGS[sample_name])
